### README

ML with XGBoost

0. Data preprocessing
1. Base model
2. Hyperparameter tuning
3. Train final model
4. Interprete feature importance 

### Requirements

In [ ]:
# Python libraries
"""
joblib==1.5.1
pandas==2.2.3
"""

In [ ]:
# Import libraries

import pandas as pd
import joblib

### Constants

In [ ]:
# Input: trained model path
MODEL_PATH = '/PATH/TO/xgb_model.pkl'

# Input: data of interest path
DATA_PATH = '/PATH/TO/features.csv'

# Output: set directory
OUTPUT_DIR =  '/PATH/TO/OUTPUT'
OUTPUT_PREFIX = ''

# Set group of features to be used
FEATURES = '3_BP' # choose between: 'all', '3_BP' (3' splice site and branch point related features) or '5_LC' (5' splice site and local context related features)

# List with values to be treated as categories
CATEGORY_GROUPS = [('A', 'C', 'G', 'U'), (0, 1)]     

### Functions

In [ ]:
#--------------------------------------------------------------------------------------------------
# Select features to be used in model training based on mode
#--------------------------------------------------------------------------------------------------

def select_features(df, mode='all'):
    
    # select features based on mode
    if mode == 'all':
        selected_cols = ['5end_-5_base', '5end_-4_base', '5end_-3_base', '5end_-2_base',
       '5end_-1_base', '5end_1_base', '5end_2_base', '5end_3_base',
       '5end_4_base', '5end_5_base', '5end_6_base', '5end_7_base',
       '3end_-7_base', '3end_-6_base', '3end_-5_base', '3end_-4_base',
       '3end_-3_base', '3end_-2_base', '3end_-1_base', '3end_1_base',
       '3end_2_base', '3end_3_base', '3end_4_base', '3end_5_base',
       '3end_6_base', '3end_7_base', '3end_8_base', 'GC(flexon)', 'GC(intron)',
       'GC(exon)', 'Len(flexon)', 'Len(intron)', 'Len(exon)',
       'proximal_3ss_score', 'distal_3ss_score', 'agez', 'ss_dist', 'bp_scr',
       'y_cont', 'ppt_off', 'ppt_len', 'ppt_scr', 'svm_scr', 'bp_same',
       'bp_1_base', 'bp_2_base', 'bp_3_base', 'bp_4_base', 'bp_5_base',
       'bp_6_base', 'bp_7_base', 'bp_8_base', 'bp_9_base']
        
    elif mode == '3_BP':
        selected_cols = ['3end_-7_base', '3end_-6_base', '3end_-5_base', '3end_-4_base',
       '3end_-3_base', '3end_-2_base', '3end_-1_base', '3end_1_base',
       '3end_2_base', '3end_3_base', '3end_4_base', '3end_5_base',
       '3end_6_base', '3end_7_base', '3end_8_base', 'proximal_3ss_score',
       'distal_3ss_score', 'agez', 'ss_dist', 'bp_scr', 'y_cont', 'ppt_off',
       'ppt_len', 'ppt_scr', 'svm_scr', 'bp_same', 'bp_1_base', 'bp_2_base',
       'bp_3_base', 'bp_4_base', 'bp_5_base', 'bp_6_base', 'bp_7_base',
       'bp_8_base', 'bp_9_base']
        
    elif mode == '5_LC':
        selected_cols = ['5end_-5_base', '5end_-4_base', '5end_-3_base', '5end_-2_base',
       '5end_-1_base', '5end_1_base', '5end_2_base', '5end_3_base',
       '5end_4_base', '5end_5_base', '5end_6_base', '5end_7_base',
       'GC(flexon)', 'GC(intron)', 'GC(exon)', 'Len(flexon)', 'Len(intron)',
       'Len(exon)']
        
    else:
        raise ValueError(f"Invalid mode type: '{mode}'.")

    return df[selected_cols].copy()

#--------------------------------------------------------------------------------------------------
# Ensure correct column data types for XGBoost with categorical features
#--------------------------------------------------------------------------------------------------

def preprocess_cols_types(df: pd.DataFrame, category_groups: list[tuple | list]):
    
    df = df.copy()

    # Pre-build categorical dtypes
    cat_dtypes = {
        tuple(group): pd.CategoricalDtype(categories=list(group))
        for group in category_groups
    }

    for col in df.columns:
        values = df[col].dropna().unique()

        # Skip empty columns
        if len(values) == 0:
            continue

        matched = False

        for group, cat_dtype in cat_dtypes.items():
            # Normalise string categories
            if all(isinstance(v, str) for v in values):
                values_norm = pd.Series(values).astype(str).str.upper().unique()
                group_norm = [str(g).upper() for g in group]
            else:
                values_norm = values
                group_norm = group

            # Check if column values fit entirely in the category group
            if set(values_norm).issubset(set(group_norm)):
                df[col] = (
                    df[col].astype(str).str.upper() # uppercase if all values in that col are strings
                    if all(isinstance(v, str) for v in values)
                    else df[col]
                )
                df[col] = df[col].astype(cat_dtype)
                matched = True
                break

        # Treat col as numerical float if values not in category groups
        if not matched:
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].astype(float)

    return df

### Analysis

In [ ]:
#---------------------------------------------------
# INPUT TABLE
#---------------------------------------------------

# read table 
input_df = pd.read_csv(DATA_PATH, sep=',', header=0)

# adjust col types and order
result_df = select_features(df=input_df, mode='3_BP')
result_df = preprocess_cols_types(result_df, category_groups=CATEGORY_GROUPS)


#---------------------------------------------------
# PREDICT EVENTS WITH ML
#---------------------------------------------------

# load model
classifier = joblib.load(MODEL_PATH)

# predict with ML
y_prob = classifier.predict_proba(result_df)[:, 1]  
y_pred = classifier.predict(result_df)

# add predictions to the dataframe
result_df['Predicted Label'] = y_pred
result_df['Predicted Probability'] = y_prob
result_df = result_df[['Predicted Label', 'Predicted Probability']].copy()

# merge predictions with original input data
result_df = result_df.merge(input_df, left_index=True, right_index=True)

result_df

In [ ]:
# save to file
result_df.to_csv(f"{OUTPUT_DIR}/{OUTPUT_PREFIX}_predictions.csv", index=False)